In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from generate_data import gen_2d_data

from diffusion_geometry import DiffusionGeometry, Function, VectorField, Form
from methods.pde import solve_differential_operator

In [ ]:



def sample_fibonacci_sphere(n: int, radius: float = 1.0) -> np.ndarray:
    indices = np.arange(n)
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle

    y = 1 - 2 * indices / (n - 1)
    r = np.sqrt(1 - y * y)
    theta = phi * indices

    x = np.cos(theta) * r
    z = np.sin(theta) * r

    points = radius * np.column_stack((x, y, z))
    return points

def spherical_frame(points: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Normalize points to unit sphere
    n = points / np.linalg.norm(points, axis=1, keepdims=True)
    x, y, z = n[:, 0], n[:, 1], n[:, 2]

    # e_phi = derivative wrt azimuthal angle φ
    e_phi = np.column_stack((-y, x, np.zeros_like(z)))
    norm_phi = np.linalg.norm(e_phi, axis=1, keepdims=True)
    # handle poles robustly
    norm_phi[norm_phi[:, 0] == 0] = 1.0
    e_phi /= norm_phi

    # e_theta = e_phi × n (right-handed frame)
    e_theta = np.cross(e_phi, n)

    return e_theta, e_phi, n


def color_by_phi(points: np.ndarray) -> np.ndarray:
    x, y = points[:, 0], points[:, 1]

    # Compute φ in [-π, π]
    phi = np.arctan2(y, x)

    # Normalize to [0, 1]
    phi_normalized = (phi + np.pi) / (2 * np.pi)

    return 2*np.pi*phi_normalized


def figure1(fig):
    n=1000
    data = sample_fibonacci_sphere(n)

    HSV_CYCLIC = [
    [0.0,  "hsv(0,100%,100%)"   ],   # red
    [0.17, "hsv(60,100%,100%)"  ],   # yellow
    [0.33, "hsv(120,100%,100%)" ],   # green
    [0.50, "hsv(180,100%,100%)" ],   # cyan
    [0.67, "hsv(240,100%,100%)" ],   # blue
    [0.83, "hsv(300,100%,100%)" ],   # magenta
    [1.0,  "hsv(360,100%,100%)" ],   # back to red
    ]

    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=1000)

    phi_color = color_by_phi(data)
    print(min(phi_color), max(phi_color))

    fig1 = plot_scatter_3d(data, color=phi_color, cyclic=True, colorscale=HSV_CYCLIC)

    fig.add_traces(list(fig1.data), rows=1, cols=1)


    e_theta, e_phi, n = spherical_frame(data)

    e_phi = VectorField.from_pointwise_basis(e_phi, dg)
    phi_op = e_phi.operator
    n_steps = 10

    print(dg.cache.u.shape)
    X = e_phi(Function.from_pointwise_basis(dg.cache.u.T, dg)).coeffs.T
    
    phi_func = Function.from_pointwise_basis(phi_color, dg)
    phi_coeffs = phi_func.coeffs

    print(phi_coeffs.shape)

    from scipy.linalg import expm

    t_int = np.linspace(0, 1, n_steps)

    for i in range(n_steps):

        t=t_int[i]
        y = expm(t * X) @ phi_coeffs

        phi_t = Function.from_coeffs(y, dg)


        fig_i = plot_scatter_3d(data, color=phi_t.to_ambient(), cyclic=True, colorscale=HSV_CYCLIC)
        fig.add_traces(list(fig_i.data), rows=1 + (i ) // 5, cols=1 + (i ) % 5)
    
    



num_rows = 2
num_cols = 5

specs = specs = [
    [{"type": "scene"}]*num_cols,  
    [{"type": "scene"}]*num_cols,   
]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=True,
    shared_yaxes=True
)

figure1(fig)

camera = dict(
    eye=dict(x=0, y=5, z=0),  # directly above (along +z)
    up=dict(x=0, y=0, z=1),     # y-axis points upward in the view
    center=dict(x=0, y=0, z=0)
)


for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(
            camera=camera,
            aspectmode="data",
        )
    })

fig.update_layout(width=1200, height=800) 


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig.show()



In [ ]:
import sys
import os


sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from generate_data import gen_2d_data
from diffusion_geometry import DiffusionGeometry, Function, VectorField, Form
from methods.pde import solve_differential_operator



def sample_perturbed_grid(range_x, range_y, num_x, num_y, noise_strength=0):
    x_lin = np.linspace(range_x[0], range_x[1], num_x)
    y_lin = np.linspace(range_y[0], range_y[1], num_y)
    x_grid, y_grid = np.meshgrid(x_lin, y_lin)
    data = np.column_stack([x_grid.ravel(), y_grid.ravel()])
    data += noise_strength * np.random.randn(*data.shape)
    return data

def figure1(fig):
    num_x = 30
    num_y = 30
    x_lim = [-10, 10]
    y_lim = [-10, 10]
    
    data = sample_perturbed_grid(range_x=x_lim, range_y=y_lim, num_x=num_x, num_y=num_y, noise_strength=0.0)

    
    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=100)

    X_coeffs = np.zeros((data.shape[0], 2))
    x, y = data[:, 0], data[:, 1]
    X_coeffs[:, 0] = -np.sin(y)
    X_coeffs[:, 1] = np.sin(x)

    scale = 1
    line_width = 1
    arrow_scale = 0.4

    X = VectorField.from_pointwise_basis(X_coeffs, dg)


  

    # fig1 = plot_scatter_2d(data)
    # fig.add_traces(list(fig1.data), rows=1, cols=1)

    fig1 = plot_quiver_2d(data, X.to_ambient(), scale=scale, line_width=line_width, arrow_scale=arrow_scale)
    fig.add_traces(list(fig1.data), rows=1, cols=1)


    def solve_ode(X: VectorField, f: Function, ts: np.array, dg) -> list[Function]:
        from scipy.linalg import expm

        X_matrix = X(Function.from_pointwise_basis(dg.cache.u.T, dg)).coeffs.T
        
        res = []
        for t in ts:
            y = expm(t * X_matrix) @ f.coeffs

            f_t = Function.from_coeffs(y, dg)
            res.append(f_t)

        return res


    f_x_list = Function.from_pointwise_basis(x, dg)
    f_y_list = Function.from_pointwise_basis(y, dg)
    res_x = solve_ode(X, f_x_list, np.linspace(0, 1, 5), dg)
    res_y = solve_ode(X, f_y_list, np.linspace(0, 1, 5), dg)

    colors = np.linspace(0, 1, len(res_x))

    for (idx, (f_x, f_y)) in enumerate(zip(res_x, res_y)):
        points_t = np.column_stack((f_x.to_ambient(), f_y.to_ambient()))

        # color = color[res_x.index(f_x)]  # same color for all points in this timestep
        t = colors[idx]

        # sample viridis at this t
        rgb_color = sample_colorscale(viridis, t)
        

        fig_i = plot_scatter_2d(points_t, color=rgb_color)
        fig.add_traces(list(fig_i.data), rows=1, cols=2)
        


 

   
    # n_steps = 10

    # print(dg.cache.u.shape)
    # X = e_phi(Function.from_pointwise_basis(dg.cache.u.T, dg)).coeffs.T
    
    # phi_func = Function.from_pointwise_basis(phi_color, dg)
    # phi_coeffs = phi_func.coeffs

    # print(phi_coeffs.shape)

    # from scipy.linalg import expm

    # t_int = np.linspace(0, 1, n_steps)

    # for i in range(n_steps):

    #     t=t_int[i]
    #     y = expm(t * X) @ phi_coeffs

    #     phi_t = Function.from_coeffs(y, dg)


    #     fig_i = plot_scatter_3d(data, color=phi_t.to_ambient(), cyclic=True, colorscale=HSV_CYCLIC)
    #     fig.add_traces(list(fig_i.data), rows=1 + (i ) // 5, cols=1 + (i ) % 5)
    
    
import plotly.express as px

# Get Plotly's Viridis colorscale (list of [pos, color] pairs)
viridis = px.colors.sequential.Viridis


def sample_colorscale(colorscale, t):
    """
    Linearly sample a Plotly colorscale at position t ∈ [0, 1].
    """
    t = 1-t
    # colorscale is like: ['rgb(...)', 'rgb(...)', ...]
    # positions are implicitly uniform
    n = len(colorscale)
    idx = t * (n - 1)
    low = int(np.floor(idx))
    high = int(np.ceil(idx))
    frac = idx - low

    def parse_rgb(s):
    # hex format like "#440154"
        if s.startswith("#"):
            s = s.lstrip("#")
            return np.array([int(s[i:i+2], 16) for i in (0, 2, 4)])
        
    # rgb(...) format
        nums = s.strip("rgb() ").split(",")
        return np.array([int(v) for v in nums])
    c_low = parse_rgb(colorscale[low])
    c_high = parse_rgb(colorscale[high])

    c = (1 - frac) * c_low + frac * c_high
    return f"rgb({int(c[0])},{int(c[1])},{int(c[2])})"



num_rows = 1
num_cols = 3

specs = specs = [
    [{"type": "xy"}]*num_cols,  
    # [{"type": "scene"}]*num_cols,   
]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=True,
    shared_yaxes=True
)

figure1(fig)

camera = dict(
    eye=dict(x=0, y=5, z=0),  # directly above (along +z)
    up=dict(x=0, y=0, z=1),     # y-axis points upward in the view
    center=dict(x=0, y=0, z=0)
)


for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(
            camera=camera,
            aspectmode="data",
        )
    })

fig.update_layout(width=800, height=400) 


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig.show()



In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from diffusion_geometry import Function, VectorField, Form
from generate_data import gen_2d_data
from diffusion_geometry import DiffusionGeometry
from methods.pde import solve_differential_operator


# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------

def sample_perturbed_grid(range_x, range_y, num_x, num_y, noise_strength=0):
    x_lin = np.linspace(range_x[0], range_x[1], num_x)
    y_lin = np.linspace(range_y[0], range_y[1], num_y)
    x_grid, y_grid = np.meshgrid(x_lin, y_lin)
    data = np.column_stack([x_grid.ravel(), y_grid.ravel()])
    data += noise_strength * np.random.randn(*data.shape)
    return data


# Viridis colorscale from Plotly
viridis = px.colors.sequential.Viridis


def sample_colorscale(colorscale, t):
    """
    Linearly sample a Plotly colorscale at position t ∈ [0, 1].
    Supports:
    - ["#RRGGBB", "#RRGGBB", ...]
    - [(pos, "#RRGGBB"), ...]
    """

    t = 1 - t

    # NEW: normalise to list of color strings
    if isinstance(colorscale[0], (list, tuple)):
        colors = [c[1] for c in colorscale]
    else:
        colors = colorscale

    n = len(colors)
    idx = t * (n - 1)
    low = int(np.floor(idx))
    high = int(np.ceil(idx))
    frac = idx - low

    def parse_rgb(s):
        if s.startswith("#"):
            s = s.lstrip("#")
            return np.array([int(s[i:i+2], 16) for i in (0, 2, 4)])
        nums = s.strip("rgb() ").split(",")
        return np.array([int(v) for v in nums])

    c_low = parse_rgb(colors[low])
    c_high = parse_rgb(colors[high])
    c = (1 - frac) * c_low + frac * c_high

    return f"rgb({int(c[0])},{int(c[1])},{int(c[2])})"



# ------------------------------------------------------------
# Main figure
# ------------------------------------------------------------

def figure1(fig):
    num_x = 30
    num_y = 30
    x_lim = [-10, 10]
    y_lim = [-10, 10]
    
    data = sample_perturbed_grid(
        range_x=x_lim, range_y=y_lim,
        num_x=num_x, num_y=num_y,
        noise_strength=0.0
    )

    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=900)

    # Vector field
    X_coeffs = np.zeros((data.shape[0], 2))
    x, y = data[:, 0], data[:, 1]
    X_coeffs[:, 0] = -np.sin(y)
    X_coeffs[:, 1] = np.sin(x)

    X = VectorField.from_pointwise_basis(X_coeffs, dg)

    # Plot quiver
    fig1 = plot_quiver_2d(data, X.to_ambient(),
                          scale=1, line_width=1, arrow_scale=0.4)
    fig.add_traces(list(fig1.data), rows=1, cols=1)

    # Solve ODE in eigenbasis
    def solve_ode(X: VectorField, f: Function, ts: np.array, dg):
        from scipy.linalg import expm
        X_matrix = X(Function.from_pointwise_basis(dg.cache.u.T, dg)).coeffs.T
        
        res = []
        for t in ts:
            y = expm(t * X_matrix) @ f.coeffs
            f_t = Function.from_coeffs(y, dg)
            res.append(f_t.to_ambient())
        return res

    f_x_list = Function.from_pointwise_basis(x, dg)
    f_y_list = Function.from_pointwise_basis(y, dg)

    num_steps = 10
    final_time = 1.5
    ts = np.linspace(0, final_time, num_steps)
    res_x = solve_ode(X, f_x_list, ts, dg)
    res_y = solve_ode(X, f_y_list, ts, dg)

    res_x = np.array(res_x).T
    res_y = np.array(res_y).T

    mycolorscale=[
            [0.0, "#00a2ff"],
            [1.0, "#ff7f7f"]
            ]

    for i in range(res_x.shape[0]):

        x_traj = res_x[i]
        y_traj = res_y[i]

        # scalar values along the trajectory (for coloring)
        tvals = np.linspace(0, 1, num_steps)

        # build line segments
        for k in range(num_steps - 1):
            segment_x = [x_traj[k], x_traj[k+1]]
            segment_y = [y_traj[k], y_traj[k+1]]

            # color for this segment
            c = tvals[k]

            seg = go.Scatter(
                x=segment_x,
                y=segment_y,
                mode="lines",
                line=dict(
                    width=2,
                    color=sample_colorscale(mycolorscale, c)
                ),
                hoverinfo="skip",
                showlegend=False
            )

            fig.add_trace(seg, row=1, col=2)

    # fig_dots = plot_scatter_2d(data, color='black', size=2)
    # fig.add_traces(list(fig_dots.data), rows=1, cols=2)
    


# ------------------------------------------------------------
# Build final plot
# ------------------------------------------------------------

num_rows = 1
num_cols = 2

specs = [
    [{"type": "xy"}] * num_cols
]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.1,
    vertical_spacing=0.05,
    shared_xaxes=True,
    shared_yaxes=True
)

figure1(fig)

camera = dict(
    eye=dict(x=0, y=5, z=0),
    up=dict(x=0, y=0, z=1),
    center=dict(x=0, y=0, z=0)
)

for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(camera=camera, aspectmode="data")
    })

fig.update_layout(width=800, height=400)

clean_fig(fig)

for r in range(num_rows):
    for c in range(num_cols):
        fig.update_xaxes(scaleanchor=f"y{c+1}", scaleratio=1, row=r+1, col=c+1)
        fig.update_yaxes(scaleanchor=f"x{c+1}", scaleratio=1, row=r+1, col=c+1)


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))


fig.write_image("figs/integral_curve.pdf", scale=1)

fig.show()


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from diffusion_geometry import Function, VectorField
from diffusion_geometry import DiffusionGeometry


# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------

def sample_perturbed_grid(range_x, range_y, num_x, num_y, noise_strength=0):
    x_lin = np.linspace(range_x[0], range_x[1], num_x)
    y_lin = np.linspace(range_y[0], range_y[1], num_y)
    x_grid, y_grid = np.meshgrid(x_lin, y_lin)
    data = np.column_stack([x_grid.ravel(), y_grid.ravel()])
    data += noise_strength * np.random.randn(*data.shape)
    return data


# ------------------------------------------------------------
# Main figure
# ------------------------------------------------------------

def figure1(fig):
    num_x = 30
    num_y = 30
    x_lim = [-10, 10]
    y_lim = [-10, 10]

    # Grid data
    data = sample_perturbed_grid(x_lim, y_lim, num_x, num_y)

    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=900)

    # Vector field
    X_coeffs = np.zeros((data.shape[0], 2))
    x, y = data[:, 0], data[:, 1]
    X_coeffs[:, 0] = -np.sin(y)
    X_coeffs[:, 1] = np.sin(x)

    X = VectorField.from_pointwise_basis(X_coeffs, dg)

    # Plot quiver
    fig1 = plot_quiver_2d(data, X.to_ambient(), scale=1, line_width=1, arrow_scale=0.4)
    fig.add_traces(list(fig1.data), rows=1, cols=1)

    # Solve ODE
    def solve_ode(X, f, ts):
        from scipy.linalg import expm
        X_matrix = X(Function.from_pointwise_basis(dg.cache.u.T, dg)).coeffs.T
        res = []
        for t in ts:
            y = expm(t * X_matrix) @ f.coeffs
            f_t = Function.from_coeffs(y, dg)
            res.append(f_t.to_ambient())
        return res

    f_x = Function.from_pointwise_basis(x, dg)
    f_y = Function.from_pointwise_basis(y, dg)

    num_steps = 100
    final_time = 1.5
    ts = np.linspace(0, final_time, num_steps)

    res_x = solve_ode(X, f_x, ts)
    res_y = solve_ode(X, f_y, ts)

    # Convert to (num_points, num_steps)
    res_x = np.array(res_x).T
    res_y = np.array(res_y).T

    # Time parameter for markers
    tvals = np.linspace(0, 1, num_steps)

    # --------------------------------------------------------
    # Draw trajectories using markers (gradient coloring)
    # --------------------------------------------------------
    for i in range(res_x.shape[0]):

        x_traj = res_x[i]
        y_traj = res_y[i]

        # Optional thin black line for continuity
        fig.add_trace(
            go.Scatter(
                x=x_traj,
                y=y_traj,
                mode="lines",
                line=dict(color="black", width=1),
                showlegend=False,
                hoverinfo="skip"
            ),
            row=1,
            col=2
        )

        # Gradient markers
        fig.add_trace(
            go.Scatter(
                x=x_traj,
                y=y_traj,
                mode="markers",
                marker=dict(
                    color=tvals,            # gradient
                    colorscale="Viridis",
                    size=2
                ),
                hoverinfo="skip",
                showlegend=False
            ),
            row=1,
            col=2
        )


# ------------------------------------------------------------
# Build final plot
# ------------------------------------------------------------

num_rows = 1
num_cols = 2

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=[[{"type": "xy"}] * num_cols],
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=True,
    shared_yaxes=True
)

figure1(fig)

fig.update_layout(
    width=900,
    height=400,
    margin=dict(l=0, r=0, t=0, b=0)
)

clean_fig(fig)

for r in range(num_rows):
    for c in range(num_cols):
        fig.update_xaxes(scaleanchor=f"y{c+1}", scaleratio=1, row=r+1, col=c+1)
        fig.update_yaxes(scaleanchor=f"x{c+1}", scaleratio=1, row=r+1, col=c+1)

fig.show()


In [108]:
# sample circle
def sample_circle(num_points, radius=1.0):
    angles = np.linspace(0, 2 * np.pi, num_points, endpoint=False)
    x = radius * np.cos(angles)
    y = radius * np.sin(angles)
    return np.column_stack((x, y))

def sample_figure8(num_points, scale=10.0):
    """
    Samples a figure-8 (Bernoulli lemniscate) curve.
    scale = overall size (similar to your radius=10 circle).
    """
    t = np.linspace(0, 2 * np.pi, num_points, endpoint=False)

    # Bernoulli lemniscate parametric equations
    x = scale * np.sin(t)
    y = scale * np.sin(t) * np.cos(t)

    return np.column_stack((x, y))


mycolorscale=[
            # [0.0, "#001f8f"],
            [0.0, "#00a2ff"],
            # [0.66, "#fefefe"],
            [1.0, "#ff7f7f"]
            ]


def draw_trajectories(X, data, dg, ts, row, col, direction=1):
        if direction == -1:
            X = -X


        def solve_ode(X, f, ts):
            from scipy.linalg import expm
            X_matrix = X(Function.from_pointwise_basis(dg.cache.u.T, dg)).coeffs.T
            res = []
            for t in ts:
                y = expm(t * X_matrix) @ f.coeffs
                f_t = Function.from_coeffs(y, dg)
                res.append(f_t.to_ambient())
            return res
        
        f_x = Function.from_pointwise_basis(data[:, 0], dg)
        f_y = Function.from_pointwise_basis(data[:, 1], dg)

        res_x = solve_ode(X, f_x, ts)
        res_y = solve_ode(X, f_y, ts)

        res_x = np.array(res_x).T
        res_y = np.array(res_y).T

        # Time parameter for markers
        tvals = np.linspace(0, 1, len(ts))

        # --------------------------------------------------------
        # Draw trajectories using markers (gradient coloring)
        # --------------------------------------------------------
        for i in range(res_x.shape[0]):

            x_traj = res_x[i]
            y_traj = res_y[i]

            # Optional thin black line for continuity
            fig.add_trace(
                go.Scatter(
                    x=x_traj,
                    y=y_traj,
                    mode="lines",
                    line=dict(color="black", width=1),
                    showlegend=False,
                    hoverinfo="skip"
                ),
                row=row,
                col=col
            )

           

            # Gradient markers
            fig.add_trace(
                go.Scatter(
                    x=x_traj,
                    y=y_traj,
                    mode="markers",
                    marker=dict(
                        color=tvals,            # gradient
                        colorscale=mycolorscale,
                        size=4
                    ),
                    hoverinfo="skip",
                    showlegend=False
                ),
                row=row,
                col=col
            )

def figure1(fig):
    n = 42
    data = sample_figure8(n, scale=10.0)
    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=100, knn_kernel=3)

    next_points = np.roll(data, shift=-1, axis=0)

    # Vector from each point to the next
    X_coeffs = next_points - data   # shape (n, 2)

    fig1 = plot_quiver_2d(data, X_coeffs, scale=0.2, line_width=1, arrow_scale=0.00001)
    fig.add_traces(list(fig1.data), rows=1, cols=1)
    fig1 = plot_quiver_2d(data, -X_coeffs, scale=0.2, line_width=1, arrow_scale=0.00001)
    fig.add_traces(list(fig1.data), rows=1, cols=1)


    X = VectorField.from_pointwise_basis(X_coeffs, dg)

    G = dg.cache.gamma_coords

    vals, vecs = np.linalg.eigh(G)

    tangent_estimate = vecs[:, :, 1]

    tangent_fig = plot_quiver_2d(
        data,
        tangent_estimate * 1,
        scale=0.5,
        line_width=2,
        arrow_scale=0.00001,
        color="red"
    )

    # fig.add_traces(list(tangent_fig.data), rows=1, cols=1)
    
    dots_fig = plot_scatter_2d(data, color="black", size=3.5)
    fig.add_traces(list(dots_fig.data), rows=1, cols=1)

    num_steps = 10
    final_time = 0.77
    ts = np.linspace(0, final_time, num_steps)

    draw_trajectories(X, data, dg, ts, row=1, col=2, direction=1)
    draw_trajectories(X, data, dg, ts, row=1, col=2, direction=-1)

    fig_dots = plot_scatter_2d(data, color='black', size=4)
    fig.add_traces(list(fig_dots.data), rows=1, cols=2)
    # fig_dots = plot_scatter_2d(data, color=mycolorscale[0][1], size=6)
    # fig.add_traces(list(fig_dots.data), rows=1, cols=2)

   
    

    








# ------------------------------------------------------------
# Build final plot
# ------------------------------------------------------------

num_rows = 1
num_cols = 2

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=[[{"type": "xy"}] * num_cols],
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=True,
    shared_yaxes=True
)

figure1(fig)

fig.update_layout(
    width=900,
    height=400,
    margin=dict(l=0, r=0, t=0, b=0)
)

clean_fig(fig)

for r in range(num_rows):
    for c in range(num_cols):
        fig.update_xaxes(scaleanchor=f"y{c+1}", scaleratio=1, row=r+1, col=c+1)
        fig.update_yaxes(scaleanchor=f"x{c+1}", scaleratio=1, row=r+1, col=c+1)

#save 
fig.write_image("figs/figure8_integral_curves.pdf", scale=1)

fig.show()

In [ ]:
def sample_torus(n_theta: int, n_phi: int, R: float = 2.0, r: float = 0.7) -> np.ndarray:
    theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
    phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
    theta, phi = np.meshgrid(theta, phi, indexing='ij')

    x = (R + r * np.cos(theta)) * np.cos(phi)
    y = (R + r * np.cos(theta)) * np.sin(phi)
    z = r * np.sin(theta)

    data = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    return data, theta.ravel(), phi.ravel()

def figure1(fig):

    # 20, 40
    data = sample_torus(n_theta=20, n_phi=40, R=2.0, r=0.7)[0]

    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=400, knn_kernel=32)


    figure1 = plot_scatter_3d(data)
    fig.add_traces(list(figure1.data), rows=1, cols=1)


    G = dg.cache.gamma_coords

    vals, vecs = np.linalg.eigh(G)

    def plot_tangent_vector(i):
        tangent_estimate = vecs[:, :, i]
        tangent_fig = plot_quiver_3d(
            data,
            tangent_estimate * 1,
            scale=0.1,
            line_width=2,
        )
        fig.add_traces(list(tangent_fig.data), rows=1, cols=1)

    plot_tangent_vector(1)
    plot_tangent_vector(2)

    def solve_ode(X, f, ts):
        from scipy.linalg import expm
        X_matrix = X(Function.from_pointwise_basis(dg.cache.u.T, dg)).coeffs.T
        res = []
        for t in ts:
            y = expm(t * X_matrix) @ f.coeffs
            f_t = Function.from_coeffs(y, dg)
            res.append(f_t.to_ambient())
        return np.array(res)
    
    n_steps = 10
    final_time = 0.1
    ts = np.linspace(0, final_time, n_steps)
    

    f_x = Function.from_pointwise_basis(data[:, 0], dg)
    f_y = Function.from_pointwise_basis(data[:, 1], dg)
    f_z = Function.from_pointwise_basis(data[:, 2], dg)

    

    def add_trajectories(X, v):
        X = VectorField.from_pointwise_basis(v, dg)

        res_x = solve_ode(X, f_x, ts)
        res_y = solve_ode(X, f_y, ts)
        res_z = solve_ode(X, f_z, ts)

        res = np.stack([res_x, res_y, res_z], axis=-1)
        res = res.reshape(-1, 3)
        return res
    

    v1 = vecs[:, :, 1]
    v2 = vecs[:, :, 2]

    new_points = []
    for v in [v1, v2, -v1, -v2]:
        X = VectorField.from_pointwise_basis(v, dg)
        res = add_trajectories(X, v)
        new_points.append(res)

    new_points = np.array(new_points).reshape(-1, 3)
    

    figure2 = plot_scatter_3d(new_points, color="black", size=1)
    fig.add_traces(list(figure2.data), rows=1, cols=2)




    

# ------------------------------------------------------------
# Build final plot
# ------------------------------------------------------------


num_rows = 1
num_cols = 2

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=[[{"type": "scene"}] * num_cols],
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=True,
    shared_yaxes=True
)

figure1(fig)

camera = dict(
    eye=dict(x=0, y=1.5, z=1),  
    up=dict(x=0, y=0, z=1),     
    center=dict(x=0, y=0, z=0)
)

for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(
            camera=camera,
            aspectmode="data",
            # xaxis=dict(**axis_range, title='x'),
            # yaxis=dict(**axis_range, title='y'),
            # zaxis=dict(range=[-2,2], title='z'),
        )
    })



fig.update_layout(width=900, height=400) 


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=10))

fig.write_image("figs/integral_curve_torus.pdf", scale=1)

fig.show()


In [73]:
def sample_torus(n_theta: int, n_phi: int, R: float = 2.0, r: float = 0.7) -> np.ndarray:
    theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
    phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
    theta, phi = np.meshgrid(theta, phi, indexing='ij')

    x = (R + r * np.cos(theta)) * np.cos(phi)
    y = (R + r * np.cos(theta)) * np.sin(phi)
    z = r * np.sin(theta)

    data = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    return data, theta.ravel(), phi.ravel()


def solve_ode(X, f, ts, dg):
        from scipy.linalg import expm
        X_matrix = X(Function.from_pointwise_basis(dg.cache.u.T, dg)).coeffs.T
        res = []
        for t in ts:
            y = expm(t * X_matrix) @ f.coeffs
            f_t = Function.from_coeffs(y, dg)
            res.append(f_t.to_ambient())
        return np.array(res)

def calculate_trajectories(X, v, ts, dg, f_x, f_y, f_z):
    X = VectorField.from_pointwise_basis(v, dg)

    res_x = solve_ode(X, f_x, ts, dg)
    res_y = solve_ode(X, f_y, ts, dg)
    res_z = solve_ode(X, f_z, ts, dg)

    res = np.stack([res_x, res_y, res_z], axis=-1)
    # res = res.reshape(-1, 3)
    return res

def iterative_add(data, ts):
    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=400)

    G = dg.cache.gamma_coords

    vals, vecs = np.linalg.eigh(G)

    f_x = Function.from_pointwise_basis(data[:, 0], dg)
    f_y = Function.from_pointwise_basis(data[:, 1], dg)
    f_z = Function.from_pointwise_basis(data[:, 2], dg)


    v1 = vecs[:, :, 1]
    v2 = vecs[:, :, 2]

    new_points = []
    for v in [v1, v2, -v1, -v2]:
        X = VectorField.from_pointwise_basis(v, dg)
        res = calculate_trajectories(X, v, ts, dg, f_x, f_y, f_z)
        new_points.append(res)

    new_points = np.array(new_points)

    last_points = new_points[:, -1, :, :].reshape(-1, 3)

    return last_points
 



def figure1(fig):

    data = sample_torus(n_theta=20, n_phi=40, R=2.0, r=0.7)[0]
    # data = sample_torus(n_theta=10, n_phi=10, R=2.0, r=0.7)[0]

    n_timesteps = 10
    final_time = 0.15

    res = iterative_add(data, ts=np.linspace(0, final_time, n_timesteps))

    marker_size = 1.8

    figure1 = plot_scatter_3d(data, color="grey", size=marker_size)
    fig.add_traces(list(figure1.data), rows=1, cols=1)
    fig.add_traces(list(figure1.data), rows=1, cols=2)
    

    figure2 = plot_scatter_3d(res, color="grey", size=marker_size)
    fig.add_traces(list(figure2.data), rows=1, cols=2)
    

    new_data = np.vstack([data, res])
    print(new_data.shape)

    res = iterative_add(new_data, ts=np.linspace(0, final_time, n_timesteps))

    figure3 = plot_scatter_3d(res, color="grey", size=marker_size)
    fig.add_traces(list(figure3.data), rows=1, cols=3)
    fig.add_traces(list(figure2.data), rows=1, cols=3)
    fig.add_traces(list(figure1.data), rows=1, cols=3)

    
# ------------------------------------------------------------
# Build final plot
# ------------------------------------------------------------


num_rows = 1
num_cols = 3

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=[[{"type": "scene"}] * num_cols],
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=True,
    shared_yaxes=True
)

figure1(fig)

camera = dict(
    eye=dict(x=0, y=1.9, z=1.1),  
    up=dict(x=0, y=0, z=1),     
    center=dict(x=0, y=0, z=0)
)

for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(
            camera=camera,
            aspectmode="data",
            # xaxis=dict(**axis_range, title='x'),
            # yaxis=dict(**axis_range, title='y'),
            # zaxis=dict(range=[-2,2], title='z'),
        )
    })



fig.update_layout(width=1200, height=400) 


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=10))
fig.show()

#save 
fig.write_image("figs/integral_curve_torus_iterative.pdf", scale=1)


(4000, 3)
